## Stroke Risk Assessment EDA and Cleaning

In [3]:
import pandas as pd

# read the csv and store in pandas data frame
data=pd.read_csv("healthcare-dataset-stroke-data.csv")

# make column names consistent
data=data.rename(columns={"Residence_type":"residence_type"})
data

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


Check for missing values

In [14]:
missing_value_proportion=data.isnull().sum()/len(data)*100
missing_value_proportion=missing_value_proportion.sort_values(ascending=False)
incomplete_columns=data.columns[data.isnull().any()]
missing_values= data[incomplete_columns]

print(f"Column/s with missing values: {incomplete_columns}")
print(missing_value_proportion["bmi"])

Column/s with missing values: Index(['bmi'], dtype='str')
3.9334637964774952


Only the bmi column has missing values

In [13]:
# create a new column indicating if bmi is missing or not
data["bmi_missing"]=data["bmi"].isna().astype(int)

# encode categorical features
data["ever_married"]=data["ever_married"].map({'Yes': 1, 'No': 0})
data["residence_type"]=data["residence_type"].map({"Urban": 1, "Rural": 0})

# one-hot encode nominal features
data_encoded=pd.get_dummies(data, columns=["gender", "work_type", "smoking_status"], drop_first=True)

# calculate the correlation between missing bmi and other numeric variables
correlation = data_encoded.drop(columns=["bmi"]).corr()["bmi_missing"].sort_values()
print(correlation)

id                               -0.127634
smoking_status_never smoked      -0.071763
work_type_children               -0.032530
work_type_Never_worked           -0.013306
gender_Other                     -0.002831
work_type_Private                -0.002144
residence_type                    0.007828
work_type_Self-employed           0.032339
smoking_status_formerly smoked    0.035087
ever_married                      0.036266
gender_Male                       0.042529
smoking_status_smokes             0.058410
age                               0.078956
avg_glucose_level                 0.091957
hypertension                      0.093046
heart_disease                     0.098621
stroke                            0.141238
bmi_missing                       1.000000
Name: bmi_missing, dtype: float64


No features have high correlation with missing bmi, meaning it is missing completely at random, so impute with median

In [17]:
# impute missing bmi's with median in both original and encoded data frame
data["bmi"]=data["bmi"].fillna(data["bmi"].median())
data_encoded["bmi"]=data["bmi"].fillna(data["bmi"].median())

Correlation

In [38]:
import plotly.express as px

# update column names for readability
data_encoded=data_encoded.rename(columns={"smoking_status_formerly smoked":"former_smoker",
                                          "smoking_status_smokes":"smoker",
                                          "work_type_Self-employed":"self_employed",
                                          "work_type_Never_worked":"never_worked",
                                          "gender_Male":"male",
                                          "avg_glucose_level":"glucose_level",
                                          "smoking_status_never smoked":"never_smoked",})

# calculate correlations
correlation=data_encoded.corr()

# create correlation heatmap
correlation_heatmap=px.imshow(correlation, color_continuous_scale="RdBu_r", height=600, width=800)
correlation_heatmap.show()


No extreme correlations between features, leave as is

Scaling and Variance

In [45]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

# scale data
scaler=StandardScaler()
scaled_data=scaler.fit_transform(data_encoded.dropna())

# find features with variance below the threshold
selector=VarianceThreshold(threshold=0.1)
selector.fit(scaled_data)

# get the feature names with low variance
low_variance_feats=data_encoded.columns[~selector.get_support()].tolist()
print(f"Low variance features: {len(low_variance_feats)}")

Low variance features: 0


No features with near-zero variance

Write cleanded and encoded dat to a csv

In [52]:
data_encoded.to_csv("stroke_data_cleaned.csv", index=False)